# FLUX.1-dev ONNX graph compatibility inspector

This notebook downloads **only small ONNX graph files** from the official gated FLUX.1-dev ONNX repository; it does not download multi-gigabyte external weight shards. It inspects transformer precision variants, graph operators, custom domains, opsets, input/output shapes, and external tensor references, then compares operator names with ONNX Runtime WebGPU's published support table.

Before running, accept the access conditions at [black-forest-labs/FLUX.1-dev-onnx](https://huggingface.co/black-forest-labs/FLUX.1-dev-onnx). This is not a generation notebook.

In [ ]:
%pip -q install -U onnx huggingface_hub requests


In [ ]:
from getpass import getpass
from huggingface_hub import login, HfApi, hf_hub_download
import os, re, json, collections, requests
import onnx

token = getpass('Hugging Face read token: ')
login(token=token, add_to_git_credential=False)
del token

REPO = 'black-forest-labs/FLUX.1-dev-onnx'
api = HfApi()
info = api.model_info(REPO, files_metadata=True)
siblings = [f for f in info.siblings if f.rfilename.endswith('.onnx')]
print('ONNX graph files:', len(siblings))
for item in sorted(siblings, key=lambda x: x.rfilename):
    print(f'{item.rfilename:65} {((item.size or 0) / 1024**2):8.2f} MiB')


## Inspect transformer precision variants

The official repository exposes BF16, FP8, and FP4 transformer directories. Only the graph files are downloaded here. External data files are intentionally left on Hugging Face.

In [ ]:
from collections import Counter, defaultdict

graph_files = [f for f in siblings if f.rfilename.startswith('transformer.opt/') and f.rfilename.endswith('/model.onnx')]
if not graph_files:
    raise RuntimeError('No transformer.opt/*/model.onnx graph files were found. Inspect the file listing above and adjust the path pattern.')

graph_results = {}
for item in sorted(graph_files, key=lambda x: x.rfilename):
    local_path = hf_hub_download(REPO, filename=item.rfilename)
    model = onnx.load(local_path, load_external_data=False)
    ops = Counter((node.domain or 'ai.onnx', node.op_type) for node in model.graph.node)
    types = Counter(onnx.TensorProto.DataType.Name(t.data_type) for t in model.graph.initializer)
    external = []
    for tensor in model.graph.initializer:
        if tensor.data_location == onnx.TensorProto.EXTERNAL:
            entry = {x.key: x.value for x in tensor.external_data}
            external.append(entry)
    graph_results[item.rfilename] = {
        'model': model,
        'ops': ops,
        'types': types,
        'external': external,
    }
    print('\n' + '=' * 88)
    print(item.rfilename, '| graph file', round((item.size or 0) / 1024**2, 2), 'MiB')
    print('IR version:', model.ir_version)
    print('Opsets:', [(x.domain or 'ai.onnx', x.version) for x in model.opset_import])
    print('Node count:', len(model.graph.node))
    print('Operator counts:')
    for (domain, op), count in sorted(ops.items()):
        print(f'  {domain:24} {op:30} {count:6}')
    print('Initializer data types:', dict(types))
    print('External initializers:', len(external))
    print('External data files:', dict(Counter(x.get('location', 'unknown') for x in external)))
    for value in list(model.graph.input)[:12]:
        dims = [d.dim_value if d.dim_value else (d.dim_param or '?') for d in value.type.tensor_type.shape.dim]
        print('Input:', value.name, onnx.TensorProto.DataType.Name(value.type.tensor_type.elem_type), dims)
    for value in list(model.graph.output)[:12]:
        dims = [d.dim_value if d.dim_value else (d.dim_param or '?') for d in value.type.tensor_type.shape.dim]
        print('Output:', value.name, onnx.TensorProto.DataType.Name(value.type.tensor_type.elem_type), dims)


## Compare operator names with ONNX Runtime WebGPU support

This is a first-pass name comparison, not a proof of compatibility: operator domain, opset, tensor type, shape, and implementation details also matter. The actual Chrome session test is still required.

In [ ]:
support_url = 'https://raw.githubusercontent.com/microsoft/onnxruntime/main/js/web/docs/webgpu-operators.md'
response = requests.get(support_url, timeout=30)
response.raise_for_status()
support_names = set()
for line in response.text.splitlines():
    if not line.startswith('|'):
        continue
    cells = [x.strip() for x in line.strip().strip('|').split('|')]
    if cells and cells[0] and cells[0].lower() not in ('operator', '---') and not cells[0].startswith(':'):
        support_names.add(cells[0].strip('`'))

for path, result in graph_results.items():
    model = result['model']
    op_names = {node.op_type for node in model.graph.node}
    unknown = sorted(op_names - support_names)
    custom_domains = sorted({node.domain for node in model.graph.node if node.domain not in ('', 'ai.onnx')})
    print('\n', path)
    print('Operator names absent from WebGPU support table:', unknown or 'none')
    print('Custom domains requiring review:', custom_domains or 'none')

print('\nNOTE: Absence from the table means investigate; presence does not guarantee this graph runs. Some listed operators are implemented only for particular opsets, shapes, or data types.')


## Export a compact report

Share this JSON report together with the browser preflight report. It contains graph metadata, not model weights.

In [ ]:
report = {}
for path, result in graph_results.items():
    model = result['model']
    report[path] = {
        'ir_version': model.ir_version,
        'opsets': [(x.domain or 'ai.onnx', x.version) for x in model.opset_import],
        'node_count': len(model.graph.node),
        'operator_counts': {domain + '::' + op: count for (domain, op), count in result['ops'].items()},
        'initializer_types': dict(result['types']),
        'external_initializer_count': len(result['external']),
        'external_data_files': dict(Counter(x.get('location', 'unknown') for x in result['external'])),
    }
with open('flux1-onnx-browser-compatibility.json', 'w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2)[:20000])
print('\nSaved flux1-onnx-browser-compatibility.json')
